# Concept Direction Experiment Harness

Unified notebook harness for concept-direction experimentation across Gemma 2 and Gemma 3 model variants.

This notebook is intended to be executed manually or via papermill. It keeps the original nine experimental phases, but it now manages resources more aggressively by opening fresh model sessions at phase boundaries and immediately offloading retained artifacts to CPU.

## Notebook Parameters

The next code cell is tagged for papermill and should remain the single source of flat experiment parameters.

In [ ]:
# Parameters - These will be injected by papermill during parameterized runs

EXPERIMENT_NAME = "gemma3_pt_capitals_states"
EXPERIMENT_CONFIG_NAME = "manual"

MODEL_FAMILY = "gemma3"
MODEL_VARIANT = "pt"
MODEL_NAME_OVERRIDE = None
TRANSCODER_SET_OVERRIDE = None

CONCEPT_PAIR_CONFIG_PATH = "cp_capitals_states_gemma_pt.yaml"
PROMPT_OVERRIDE = "Fact: the capital of the state containing Dallas is"
PROMPT_RENDER_MODE = "plain"

TOP_N = 10
DEFAULT_SCALE_FACTOR = 10.0
SCALE_FACTOR_SWEEP = [2.0, 5.0, 10.0, 20.0, 50.0]
ABLATION_N_LIST = [5, 10, 25, 50, 100]
ENABLE_SIGN_AWARE = True

# Analysis modes:
# - "concept_pair": existing embed/store concept-direction workflow
# - "explicit_embedding_difference": use EXPLICIT_DIRECTION_TOKENS as the direct embedding-difference direction
# - "debug_intervention_pipelines": skip concept-direction phases and run single-feature intervention validation
ANALYSIS_MODE = "concept_pair"
EXPLICIT_DIRECTION_TOKENS = None
ENABLE_ZERO_SOFTCAP = False

# Session surface presets:
# - "notebook_default": preserve the historical notebook session construction
# - "parity_surface": align notebook sessions with the passing standalone parity surface
#   while preserving the configured or auto-selected device
DEBUG_SESSION_SURFACE_PRESET = "notebook_default"

# Debug validation tolerances follow the circuit-tracer Gemma 3 intervention-validation defaults,
# with a looser logit atol for custom-diff targets when needed.
DEBUG_VALIDATION_LOGIT_ATOL = 1e-4
DEBUG_VALIDATION_LOGIT_RTOL = 1e-3
DEBUG_VALIDATION_ACT_ATOL = 1e-3
DEBUG_VALIDATION_ACT_RTOL = 1e-5
DEBUG_VALIDATION_TOP_K = 10
DEBUG_VALIDATION_RAISE_ON_FAILURE = True

# Experiment-owned key tokens tracked in sanity checks and report displays.
KEY_TOKENS = ["▁Austin", "▁Dallas", "▁Texas", "▁Capital", "▁State", "▁capital", "▁state", "▁City", "▁city"]

# Optional constrained-feature override — one or more Neuronpedia refs limiting candidate
# intervention features before score ranking. Each entry may be a full feature URL,
# a model/layer/index ref, a layer/index ref, or a (model_id, source_set, layer, feature_index)
# tuple/list.
CONSTRAINED_FEATURE_SELECTION_LIST = None

# TARGET_TOKENS takes precedence over TARGET_TOKEN_IDS when both are provided.
TARGET_TOKENS = None
TARGET_TOKEN_IDS = (24278, 26057)

NEURONPEDIA_MODEL_OVERRIDE = None
NEURONPEDIA_SET_OVERRIDE = None
USE_LOCALHOST = False
NEURONPEDIA_BASE_URL_OVERRIDE = None
LOCAL_NEURONPEDIA_DB_URL = None
LOCAL_NEURONPEDIA_WEBAPP_URL = None
CHECK_LOCAL_EXPLANATION_COVERAGE = False
GENERATE_MISSING_LOCAL_EXPLANATIONS = False
LOCAL_EXPLANATION_FEATURE_LIMIT = 20
LOCAL_EXPLANATION_TYPE_NAME = "np_max-act-logits"
LOCAL_EXPLANATION_TIMEOUT_SECONDS = 120
LOCAL_EXPLANATION_MAX_RETRIES = 2
LOCAL_EXPLANATION_RETRY_BACKOFF_SECONDS = 5.0
FORCE_DEVICE = None
EXPERIMENT_WORK_DIR = None
VERBOSE = True
ENABLE_BASELINE_PATH_DEBUG = False

# Resource overrides — set via YAML configs to reduce peak VRAM for large/IT models.
# When None, cfg_aliases defaults apply (batch_size=256, max_feature_nodes=8192).
BATCH_SIZE = None
MAX_FEATURE_NODES = None

# V12 approach controls — which extraction mode and intervention type to use.
# "answer_position_state" = default (approaches a, b, c)
# "context_enhanced" = context-enhanced extraction (approaches d, e)
STORE_LATENT_EXTRACTION_MODE = "answer_position_state"
CONTEXT_ENHANCED_SCALE = 1.0

In [ ]:
from __future__ import annotations

import gc
import json
import os
import shutil
from pathlib import Path
from typing import Any

# Reduce CUDA memory fragmentation before importing torch.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import matplotlib.pyplot as plt
import torch

from notebook_bootstrap import bootstrap_notebook_imports

BOOTSTRAP_PATHS = bootstrap_notebook_imports(Path.cwd())

from interpretune.utils import ensure_local_feature_explanations
from it_examples.utils.nb_ui_utils import (
    display_ablation_chart,
    display_key_token_logits,
    display_layer_divergence_summary,
    display_logit_drift_summary,
    display_top_features_comparison,
    display_topk_token_predictions,
)
from concept_direction_experiment_utils import (
    NotebookHarnessConfig,
    collect_baseline_path_debug as _collect_baseline_path_debug,
    collect_feature_pool as _collect_feature_pool,
    compute_embed_direction as _compute_embed_direction,
    compute_store_direction as _compute_store_direction,
    phase_run_name as _phase_run_name,
    prepare_local_explanation_backfill,
    resolve_concept_pair_config_path,
    resolve_neuronpedia_runtime_config,
    run_ablations as _run_ablations,
    run_debug_intervention_validation as _run_debug_intervention_validation,
    run_direction_probes as _run_direction_probes,
    run_direct_projection_pipeline as _run_direct_projection_pipeline,
    run_direct_projection_scale_sweep as _run_direct_projection_scale_sweep,
    run_initial_sanity_check as _run_initial_sanity_check,
    run_pipeline as _run_pipeline,
    run_scale_sweep as _run_scale_sweep,
    run_sign_aware as _run_sign_aware,
    run_tokenizer_verification as _run_tokenizer_verification,
)
from experiment_resource_utils import (
    create_work_root,
    experiment_session,
    feature_ids_to_tuples,
    resolve_model_spec,
    tensor_to_cpu,
)

print("Imports complete.")

In [ ]:
MODEL_SPEC = resolve_model_spec(
    MODEL_FAMILY,
    MODEL_VARIANT,
    model_name_override=MODEL_NAME_OVERRIDE,
    transcoder_set_override=TRANSCODER_SET_OVERRIDE,
    neuronpedia_model_override=NEURONPEDIA_MODEL_OVERRIDE,
    neuronpedia_set_override=NEURONPEDIA_SET_OVERRIDE,
)

MODEL_NAME = MODEL_SPEC.model_name
TRANSCODER_SET = MODEL_SPEC.transcoder_set
HF_MODEL_HEAD = MODEL_SPEC.hf_model_head
NEURONPEDIA_MODEL = MODEL_SPEC.neuronpedia_model
NEURONPEDIA_SET = MODEL_SPEC.neuronpedia_set
WORK_ROOT = create_work_root(EXPERIMENT_WORK_DIR, EXPERIMENT_NAME)

NEURONPEDIA_RUNTIME = resolve_neuronpedia_runtime_config(
    use_localhost=USE_LOCALHOST,
    neuronpedia_base_url_override=NEURONPEDIA_BASE_URL_OVERRIDE,
    local_db_url=LOCAL_NEURONPEDIA_DB_URL,
    local_webapp_url=LOCAL_NEURONPEDIA_WEBAPP_URL,
    check_local_explanation_coverage=CHECK_LOCAL_EXPLANATION_COVERAGE,
    generate_missing_local_explanations=GENERATE_MISSING_LOCAL_EXPLANATIONS,
    local_explanation_feature_limit=LOCAL_EXPLANATION_FEATURE_LIMIT,
)
NEURONPEDIA_BASE_URL = NEURONPEDIA_RUNTIME.base_url
CONCEPT_PAIR_PATH = resolve_concept_pair_config_path(
    None,
    model_family=MODEL_FAMILY,
    prompt_render_mode=PROMPT_RENDER_MODE,
    concept_pair_config_path=CONCEPT_PAIR_CONFIG_PATH,
)
if PROMPT_OVERRIDE is None:
    raise ValueError("PROMPT_OVERRIDE must be provided by the experiment config.")

NOTEBOOK_CFG = NotebookHarnessConfig(
    experiment_name=EXPERIMENT_NAME,
    experiment_config_name=EXPERIMENT_CONFIG_NAME,
    model_family=MODEL_FAMILY,
    model_variant=MODEL_VARIANT,
    model_name=MODEL_NAME,
    transcoder_set=TRANSCODER_SET,
    hf_model_head=HF_MODEL_HEAD,
    neuronpedia_model=NEURONPEDIA_MODEL,
    neuronpedia_set=NEURONPEDIA_SET,
    neuronpedia_base_url=NEURONPEDIA_BASE_URL,
    concept_pair_name=None,
    concept_pair_config_path=str(CONCEPT_PAIR_PATH),
    prompt=PROMPT_OVERRIDE,
    prompt_render_mode=PROMPT_RENDER_MODE,
    target_tokens=tuple(TARGET_TOKENS) if TARGET_TOKENS is not None else None,
    target_token_ids=tuple(TARGET_TOKEN_IDS) if TARGET_TOKEN_IDS is not None else None,
    top_n=TOP_N,
    default_scale_factor=DEFAULT_SCALE_FACTOR,
    scale_factor_sweep=SCALE_FACTOR_SWEEP,
    ablation_n_list=ABLATION_N_LIST,
    enable_sign_aware=ENABLE_SIGN_AWARE,
    force_device=FORCE_DEVICE,
    work_root=WORK_ROOT,
    batch_size=BATCH_SIZE,
    max_feature_nodes=MAX_FEATURE_NODES,
    use_localhost=NEURONPEDIA_RUNTIME.use_localhost,
    local_neuronpedia_db_url=NEURONPEDIA_RUNTIME.local_db_url,
    local_neuronpedia_webapp_url=NEURONPEDIA_RUNTIME.local_webapp_url,
    check_local_explanation_coverage=NEURONPEDIA_RUNTIME.check_local_explanation_coverage,
    generate_missing_local_explanations=NEURONPEDIA_RUNTIME.generate_missing_local_explanations,
    local_explanation_feature_limit=NEURONPEDIA_RUNTIME.local_explanation_feature_limit,
    local_neuronpedia_service_status=NEURONPEDIA_RUNTIME.service_status,
    mode_warning_messages=NEURONPEDIA_RUNTIME.warning_messages,
    local_explanation_type_name=LOCAL_EXPLANATION_TYPE_NAME,
    local_explanation_timeout_seconds=LOCAL_EXPLANATION_TIMEOUT_SECONDS,
    local_explanation_max_retries=LOCAL_EXPLANATION_MAX_RETRIES,
    local_explanation_retry_backoff_seconds=LOCAL_EXPLANATION_RETRY_BACKOFF_SECONDS,
    key_tokens_override=tuple(KEY_TOKENS) if KEY_TOKENS is not None else None,
    constrained_feature_selection_refs=(
        tuple(CONSTRAINED_FEATURE_SELECTION_LIST) if CONSTRAINED_FEATURE_SELECTION_LIST is not None else None
    ),
    store_latent_extraction_mode=STORE_LATENT_EXTRACTION_MODE,
    context_enhanced_scale=CONTEXT_ENHANCED_SCALE,
    enable_baseline_path_debug=ENABLE_BASELINE_PATH_DEBUG,
    analysis_mode=ANALYSIS_MODE,
    explicit_direction_tokens=(
        tuple(EXPLICIT_DIRECTION_TOKENS) if EXPLICIT_DIRECTION_TOKENS is not None else None
    ),
    enable_zero_softcap=ENABLE_ZERO_SOFTCAP,
    debug_session_surface_preset=DEBUG_SESSION_SURFACE_PRESET,
    debug_validation_logit_atol=DEBUG_VALIDATION_LOGIT_ATOL,
    debug_validation_logit_rtol=DEBUG_VALIDATION_LOGIT_RTOL,
    debug_validation_act_atol=DEBUG_VALIDATION_ACT_ATOL,
    debug_validation_act_rtol=DEBUG_VALIDATION_ACT_RTOL,
    debug_validation_top_k=DEBUG_VALIDATION_TOP_K,
    debug_validation_raise_on_failure=DEBUG_VALIDATION_RAISE_ON_FAILURE,
)
concept_pair = NOTEBOOK_CFG.concept_pair
ANALYSIS_MODE_NAME = NOTEBOOK_CFG.analysis_mode
USES_STORE_DIRECTION = NOTEBOOK_CFG.supports_store_direction
IS_DEBUG_INTERVENTION_MODE = NOTEBOOK_CFG.is_debug_intervention_mode

if VERBOSE:
    print(
        json.dumps(
            {
                "experiment_name": NOTEBOOK_CFG.experiment_name,
                "experiment_config_name": NOTEBOOK_CFG.experiment_config_name,
                "model_name": NOTEBOOK_CFG.model_name,
                "transcoder_set": NOTEBOOK_CFG.transcoder_set,
                "neuronpedia_model": NOTEBOOK_CFG.neuronpedia_model,
                "neuronpedia_set": NOTEBOOK_CFG.neuronpedia_set,
                "concept_pair": NOTEBOOK_CFG.concept_pair_name,
                "analysis_mode": NOTEBOOK_CFG.analysis_mode,
                "analysis_concept_label": NOTEBOOK_CFG.analysis_concept_label,
                "explicit_direction_tokens": NOTEBOOK_CFG.explicit_direction_tokens,
                "enable_zero_softcap": NOTEBOOK_CFG.enable_zero_softcap,
                "debug_session_surface_preset": NOTEBOOK_CFG.debug_session_surface_preset,
                "prompt": NOTEBOOK_CFG.prompt,
                "target_tokens": NOTEBOOK_CFG.target_tokens,
                "target_token_ids": NOTEBOOK_CFG.target_token_ids,
                "top_n": NOTEBOOK_CFG.top_n,
                "default_scale_factor": NOTEBOOK_CFG.default_scale_factor,
                "scale_factor_sweep": NOTEBOOK_CFG.scale_factor_sweep,
                "ablation_n_list": NOTEBOOK_CFG.ablation_n_list,
                "enable_sign_aware": NOTEBOOK_CFG.enable_sign_aware,
                "batch_size": NOTEBOOK_CFG.batch_size,
                "max_feature_nodes": NOTEBOOK_CFG.max_feature_nodes,
                "force_device": NOTEBOOK_CFG.force_device,
                "work_root": str(NOTEBOOK_CFG.work_root),
                "use_localhost": NOTEBOOK_CFG.use_localhost,
                "neuronpedia_base_url": NOTEBOOK_CFG.neuronpedia_base_url,
                "local_neuronpedia_db_url": NOTEBOOK_CFG.local_neuronpedia_db_url,
                "local_neuronpedia_webapp_url": NOTEBOOK_CFG.local_neuronpedia_webapp_url,
                "check_local_explanation_coverage": NOTEBOOK_CFG.check_local_explanation_coverage,
                "generate_missing_local_explanations": NOTEBOOK_CFG.generate_missing_local_explanations,
                "local_explanation_feature_limit": NOTEBOOK_CFG.local_explanation_feature_limit,
                "local_explanation_type_name": NOTEBOOK_CFG.local_explanation_type_name,
                "prompt_render_mode": NOTEBOOK_CFG.prompt_render_mode,
                "constrained_feature_selection_refs": NOTEBOOK_CFG.constrained_feature_selection_refs,
                "store_latent_extraction_mode": NOTEBOOK_CFG.store_latent_extraction_mode,
                "context_enhanced_scale": NOTEBOOK_CFG.context_enhanced_scale,
                "debug_validation_top_k": NOTEBOOK_CFG.debug_validation_top_k,
                "debug_validation_raise_on_failure": NOTEBOOK_CFG.debug_validation_raise_on_failure,
                "debug_validation_tolerances": {
                    "act_atol": NOTEBOOK_CFG.debug_validation_act_atol,
                    "act_rtol": NOTEBOOK_CFG.debug_validation_act_rtol,
                    "logit_atol": NOTEBOOK_CFG.debug_validation_logit_atol,
                    "logit_rtol": NOTEBOOK_CFG.debug_validation_logit_rtol,
                },
            },
            indent=2,
            default=str,
        )
    )

for warning_message in NOTEBOOK_CFG.mode_warning_messages:
    print(f"WARNING: {warning_message}")

In [ ]:
def phase_run_name(label: str) -> str:
    return _phase_run_name(NOTEBOOK_CFG, label)


def print_phase_skip(reason: str) -> None:
    print(f"Skipped for analysis_mode={NOTEBOOK_CFG.analysis_mode}: {reason}")


def run_tokenizer_verification() -> dict[str, Any]:
    return _run_tokenizer_verification(NOTEBOOK_CFG)


def run_initial_sanity_check() -> dict[str, Any]:
    return _run_initial_sanity_check(NOTEBOOK_CFG)


def collect_baseline_path_debug() -> dict[str, Any]:
    return _collect_baseline_path_debug(NOTEBOOK_CFG)


def compute_embed_direction() -> dict[str, Any]:
    return _compute_embed_direction(NOTEBOOK_CFG)


def run_pipeline(
    direction: torch.Tensor,
    label: str,
    *,
    scale_factor: float,
    top_n: int,
    group_a_ids: list[int],
    group_b_ids: list[int],
) -> dict[str, Any]:
    return _run_pipeline(
        NOTEBOOK_CFG,
        direction,
        label,
        scale_factor=scale_factor,
        top_n=top_n,
        group_a_ids=group_a_ids,
        group_b_ids=group_b_ids,
    )


def compute_store_direction() -> dict[str, Any]:
    return _compute_store_direction(NOTEBOOK_CFG)


def run_scale_sweep(direction: torch.Tensor, group_a_ids: list[int], group_b_ids: list[int]) -> list[dict[str, Any]]:
    return _run_scale_sweep(NOTEBOOK_CFG, direction, group_a_ids, group_b_ids)


def collect_feature_pool(
    direction: torch.Tensor,
    group_a_ids: list[int],
    group_b_ids: list[int],
    *,
    top_n: int,
) -> dict[str, Any]:
    return _collect_feature_pool(NOTEBOOK_CFG, direction, group_a_ids, group_b_ids, top_n=top_n)


def run_ablations(
    feature_pool: dict[str, Any],
    pre_logits_ref: torch.Tensor,
) -> tuple[dict[str, dict[str, float]], dict[str, float], dict[int, torch.Tensor]]:
    return _run_ablations(NOTEBOOK_CFG, feature_pool, pre_logits_ref)


def run_sign_aware(feature_pool: dict[str, Any], pre_logits_ref: torch.Tensor) -> dict[str, Any]:
    return _run_sign_aware(NOTEBOOK_CFG, feature_pool, pre_logits_ref)


def run_direct_projection_pipeline(
    direction: torch.Tensor,
    label: str,
    *,
    scale_factor: float,
) -> dict[str, Any]:
    return _run_direct_projection_pipeline(NOTEBOOK_CFG, direction, label, scale_factor=scale_factor)


def run_direct_projection_scale_sweep(direction: torch.Tensor) -> list[dict[str, Any]]:
    return _run_direct_projection_scale_sweep(NOTEBOOK_CFG, direction)


def run_direction_probes(embed_direction: torch.Tensor, store_direction: torch.Tensor) -> dict[str, Any]:
    return _run_direction_probes(NOTEBOOK_CFG, embed_direction, store_direction)


def run_debug_intervention_validation() -> dict[str, Any]:
    return _run_debug_intervention_validation(NOTEBOOK_CFG)


RESULTS: dict[str, Any] = {}

## Phase 1: Session Initialization and Tokenizer Verification

In [ ]:
tokenizer_report = run_tokenizer_verification()
RESULTS["tokenizer_report"] = tokenizer_report

print(f"Module type: {tokenizer_report['module_type']}")
print(f"Prompt render mode: {tokenizer_report['prompt_render_mode']}")
print(f"Prompt token count: {tokenizer_report['prompt_token_count']}")
print(
    "Target tokens: "
    f"A={tokenizer_report['target_tokens']['group_a']['id']} ({tokenizer_report['target_tokens']['group_a']['decoded']!r}), "
    f"B={tokenizer_report['target_tokens']['group_b']['id']} ({tokenizer_report['target_tokens']['group_b']['decoded']!r})"
)

for group_name, entries in tokenizer_report["groups"].items():
    print(f"\n{group_name}:")
    for entry in entries:
        print(f"  {entry['token']}: ids={entry['ids']}, decoded='{entry['decoded']}'")

print("\nKey tokens:")
for token, payload in tokenizer_report["key_tokens"].items():
    print(f"  {token}: ids={payload['ids']}, decoded='{payload['decoded']}'")

print("\nResolved key-token candidates:")
for entry in tokenizer_report["resolved_key_token_candidates"]:
    print(
        "  "
        f"{entry['label']} [{entry['variant']}] from {entry['source_token']}: "
        f"token_id={entry['token_id']}, ids={entry['ids']}, decoded={entry['decoded']!r}"
    )

print("\nRendered prompt variants:")
for mode_name, preview in tokenizer_report["render_variants"].items():
    if preview is None:
        print(f"\n[{mode_name}]\n(not available — model has no chat template)")
        continue
    print(f"\n[{mode_name}]\n{preview[:300]}")
    print(f"token_ids={tokenizer_report['render_variant_token_ids'][mode_name]}")
    print(f"tokens={tokenizer_report['render_variant_tokens'][mode_name][:40]}")

equalities = tokenizer_report["render_variant_equalities"]
print("\nPrompt parity checks:")
print(f"  apply_chat_template vs gemma_dataclass string parity: {equalities['apply_chat_template_vs_dataclass']}")
print(
    "  apply_chat_template vs gemma_dataclass token-id parity: "
    f"{equalities['apply_chat_template_vs_dataclass_token_ids']}"
)

print("\nSelected prompt preview:")
print(tokenizer_report["selected_prompt_preview"])
print(f"Selected prompt token ids: {tokenizer_report['selected_prompt_token_ids']}")
print(f"Selected prompt tokens: {tokenizer_report['selected_prompt_tokens'][:60]}")

In [ ]:
from it_examples.utils.nb_ui_utils import format_prob as _fp

reasoning_check = run_initial_sanity_check()
RESULTS["initial_sanity_check"] = reasoning_check

print("Initial Sanity Check")
print(f"Prompt style: {reasoning_check['prompt_style']}")
print(f"Prompt: {reasoning_check['rendered_prompt'][:200]}")
print(f"Generated: {reasoning_check['generated_text']}")

print("\nFirst-token logit analysis:")
print(f"  {'Token':<12} {'ID':>8} {'Logit':>10} {'Prob':>12}")
print(f"  {'-' * 46}")
for entry in reasoning_check["key_tokens"]:
    if entry.get("token_id") is not None:
        print(f"  {entry['label']:<12} {entry['token_id']:>8} {entry['logit']:>10.4f} {_fp(entry['prob']):>12}")
    else:
        print(f"  {entry['label']:<12} {'N/A':>8} {'N/A':>10} {'N/A':>12}")
print(f"  {'-' * 46}")
print(
    f"  {'Top-1':<12} {reasoning_check['top1_id']:>8} {reasoning_check['top1_logit']:>10.4f} {_fp(reasoning_check['top1_prob']):>12}  ({reasoning_check['top1_token']!r})"
)

In [ ]:
if NOTEBOOK_CFG.enable_baseline_path_debug:
    baseline_path_debug = collect_baseline_path_debug()
    RESULTS["baseline_path_debug"] = baseline_path_debug

    print("Baseline Path Debug")
    print(f"Prompt render mode: {baseline_path_debug['prompt_render_mode']}")
    print(f"Generation kwargs: {json.dumps(baseline_path_debug['generation_kwargs'], sort_keys=True)}")
    print(f"Raw prompt: {baseline_path_debug['prompt_debug']['raw_prompt']}")
    print(f"Rendered prompt: {baseline_path_debug['prompt_debug']['rendered_prompt']}")
    print(f"Input IDs: {baseline_path_debug['prompt_debug']['input_ids']}")
    print(f"Tokens: {baseline_path_debug['prompt_debug']['tokens'][:80]}")

    print("\nBaseline source top-1 summaries:")
    for source_name, entries in baseline_path_debug['baseline_sources'].items():
        top_entry = entries[0]
        print(
            f"  {source_name:<28} {top_entry['token']!r} "
            f"(id={top_entry['token_id']}, logit={top_entry['logit']:.4f}, prob={top_entry['prob']:.6f})"
        )

    print("\nMax abs diffs:")
    for diff_name, diff_value in baseline_path_debug['max_abs_diffs'].items():
        print(f"  {diff_name}: {diff_value:.6f}")
else:
    print("Baseline path debug: DISABLED (set ENABLE_BASELINE_PATH_DEBUG=true to enable)")

## Phase 2 and 3: Embed-Based Direction and Full Pipeline

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("embed-direction pipeline is replaced by debug intervention validation.")
else:
    embed_direction_result = compute_embed_direction()
    embed_pipeline = run_pipeline(
        embed_direction_result["direction"],
        "Embed",
        scale_factor=DEFAULT_SCALE_FACTOR,
        top_n=TOP_N,
        group_a_ids=embed_direction_result["group_a_ids"],
        group_b_ids=embed_direction_result["group_b_ids"],
    )
    RESULTS["embed_direction_result"] = embed_direction_result
    RESULTS["embed_pipeline"] = embed_pipeline
    print(f"Embed direction norm: {torch.linalg.vector_norm(embed_direction_result['direction']):.6f}")
    print(
        f"Pre-gap ({embed_pipeline['target_a_tok']}−{embed_pipeline['target_b_tok']}): "
        f"{embed_pipeline['pre_gap']:.4f}"
    )
    print(f"Post-gap: {embed_pipeline['post_gap']:.4f}")
    print(f"Gap Δ: {embed_pipeline['gap_delta']:+.4f}")
    display_top_features_comparison(
        {"Embed — Top Features": embed_pipeline["feature_ids"]},
        {"Embed — Top Features": embed_pipeline["feature_scores"]},
        neuronpedia_model=NEURONPEDIA_MODEL,
        neuronpedia_set=NEURONPEDIA_SET,
        neuronpedia_base_url=NOTEBOOK_CFG.neuronpedia_base_url,
    )

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("embed-direction display is replaced by debug intervention validation.")
else:
    with experiment_session(
        WORK_ROOT,
        phase_run_name("embed_display"),
        **NOTEBOOK_CFG.session_kwargs,
    ) as (_, _, tokenizer):
        display_topk_token_predictions(
            embed_pipeline["rendered_prompt"],
            embed_pipeline["pre_logits"],
            embed_pipeline["post_logits"],
            tokenizer,
            k=5,
            key_tokens=[
                (label, token_id) for label, token_id in zip(embed_pipeline["key_labels"], embed_pipeline["key_ids"])
            ],
        )
        display_key_token_logits(
            embed_pipeline["pre_logits"],
            embed_pipeline["post_logits"],
            embed_pipeline["key_ids"],
            embed_pipeline["key_labels"],
            title=f"Embed {DEFAULT_SCALE_FACTOR}× — Key-Token Logits",
        )

## Phase 4 and 5: Store-Based Direction and Summary

In [ ]:
if not USES_STORE_DIRECTION:
    print_phase_skip("store-direction pipeline only applies to concept_pair mode.")
else:
    store_direction_result = compute_store_direction()
    store_pipeline = run_pipeline(
        store_direction_result["direction"],
        "Store",
        scale_factor=DEFAULT_SCALE_FACTOR,
        top_n=TOP_N,
        group_a_ids=store_direction_result["group_a_ids"],
        group_b_ids=store_direction_result["group_b_ids"],
    )
    RESULTS["store_direction_result"] = store_direction_result
    RESULTS["store_pipeline"] = store_pipeline
    cosine_similarity = float(
        torch.nn.functional.cosine_similarity(
            embed_direction_result["direction"].unsqueeze(0), store_direction_result["direction"].unsqueeze(0)
        ).item()
    )
    embed_set = set(embed_pipeline["feature_ids"])
    store_set = set(store_pipeline["feature_ids"])
    shared = embed_set & store_set
    total = embed_set | store_set
    feature_jaccard = len(shared) / len(total) if total else 0.0
    RESULTS["comparison"] = {
        "cosine_similarity": cosine_similarity,
        "feature_jaccard": feature_jaccard,
        "shared_feature_count": len(shared),
        "total_feature_count": len(total),
    }
    print(f"Store direction norm: {torch.linalg.vector_norm(store_direction_result['direction']):.6f}")
    print(f"Predictions: {store_direction_result['prediction_info']['n_correct']}/{store_direction_result['n_total']}")
    print(f"Cosine similarity: {cosine_similarity:.6f}")
    print(f"Feature Jaccard: {feature_jaccard:.4f} ({len(shared)}/{len(total)})")
    print(f"Embed Δ: {embed_pipeline['gap_delta']:+.4f} | Store Δ: {store_pipeline['gap_delta']:+.4f}")

In [ ]:
if not USES_STORE_DIRECTION:
    print_phase_skip("store-direction display only applies to concept_pair mode.")
else:
    with experiment_session(
        WORK_ROOT,
        phase_run_name("store_display"),
        **NOTEBOOK_CFG.session_kwargs,
    ) as (_, _, tokenizer):
        display_top_features_comparison(
            {
                "Embed — Top Features": embed_pipeline["feature_ids"],
                "Store — Top Features": store_pipeline["feature_ids"],
            },
            {
                "Embed — Top Features": embed_pipeline["feature_scores"],
                "Store — Top Features": store_pipeline["feature_scores"],
            },
            neuronpedia_model=NEURONPEDIA_MODEL,
            neuronpedia_set=NEURONPEDIA_SET,
            neuronpedia_base_url=NOTEBOOK_CFG.neuronpedia_base_url,
        )
        display_topk_token_predictions(
            store_pipeline["rendered_prompt"],
            store_pipeline["pre_logits"],
            store_pipeline["post_logits"],
            tokenizer,
            k=5,
            key_tokens=[
                (label, token_id) for label, token_id in zip(store_pipeline["key_labels"], store_pipeline["key_ids"])
            ],
        )
        display_key_token_logits(
            store_pipeline["pre_logits"],
            store_pipeline["post_logits"],
            store_pipeline["key_ids"],
            store_pipeline["key_labels"],
            title=f"Store {DEFAULT_SCALE_FACTOR}× — Key-Token Logits",
        )

## Phase 4b: Direct Projection (Store Direction)

Bypasses feature selection entirely — adds the scaled store concept direction
vector directly to the residual stream at `unembed.hook_in` (lm_head input).

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("direct projection is replaced by debug intervention validation.")
else:
    direct_projection_source = store_direction_result if USES_STORE_DIRECTION else embed_direction_result
    direct_projection_label = "DirectProj_Store" if USES_STORE_DIRECTION else "DirectProj_Embed"
    dp_pipeline = run_direct_projection_pipeline(
        direct_projection_source["direction"],
        direct_projection_label,
        scale_factor=DEFAULT_SCALE_FACTOR,
    )
    RESULTS["dp_pipeline"] = dp_pipeline
    print(f"Direct Projection ({direct_projection_label}, {DEFAULT_SCALE_FACTOR}×):")
    print(f"  Pre gap:  {dp_pipeline['pre_gap']:+.4f}")
    print(f"  Post gap: {dp_pipeline['post_gap']:+.4f}")
    print(f"  Δ gap:    {dp_pipeline['gap_delta']:+.4f}")
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("direct projection display is replaced by debug intervention validation.")
else:
    with experiment_session(
        WORK_ROOT,
        phase_run_name("dp_display"),
        **NOTEBOOK_CFG.session_kwargs,
    ) as (_, _, tokenizer):
        display_topk_token_predictions(
            dp_pipeline["rendered_prompt"],
            dp_pipeline["pre_logits"],
            dp_pipeline["post_logits"],
            tokenizer,
            k=5,
            key_tokens=[(label, token_id) for label, token_id in zip(dp_pipeline["key_labels"], dp_pipeline["key_ids"])],
        )
        display_key_token_logits(
            dp_pipeline["pre_logits"],
            dp_pipeline["post_logits"],
            dp_pipeline["key_ids"],
            dp_pipeline["key_labels"],
            title=f"Direct Projection {DEFAULT_SCALE_FACTOR}× — Key-Token Logits",
        )

## Phase 6: Scale Factor Sweep or Debug Validation

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    debug_validation = run_debug_intervention_validation()
    RESULTS["debug_validation"] = debug_validation
    print("Debug Intervention Validation")
    print(f"Selected feature: {debug_validation['selected_feature']}")
    print(f"Selected feature score: {debug_validation['selected_feature_score']:+.6f}")
    print(f"Baseline activation: {debug_validation['baseline_activation']:+.6f}")
    print(f"Intervention value: {debug_validation['intervention_value']:+.6f}")
    print(f"Pre-gap: {debug_validation['pre_gap']:+.6f}")
    print(f"Post-gap: {debug_validation['post_gap']:+.6f}")
    print(f"Gap Δ: {debug_validation['gap_delta']:+.6f}")
    print(f"Activation check passed: {debug_validation['activation_passed']}")
    print(f"Logit check passed: {debug_validation['logit_passed']}")
    if debug_validation["artifact_dir"] is not None:
        print(f"Preserved artifact dir: {debug_validation['artifact_dir']}")

    def display_diagnostic_rows(title, rows):
        print(title)
        if not rows:
            print("  <none>")
            return
        display(rows)

    input_summary = {
        key: debug_validation["input_diagnostics"][key]
        for key in [
            "graph_input_source",
            "rendered_prompt_token_count",
            "graph_input_token_count",
            "graph_inputs_match_rendered_prompt",
            "first_difference_index",
        ]
    }
    drift_report = debug_validation["drift_report"]
    print(
        json.dumps(
            {
                "activation_max_abs_error": debug_validation["activation_max_abs_error"],
                "activation_mean_abs_error": debug_validation["activation_mean_abs_error"],
                "activation_sign_mismatch_count": debug_validation["activation_sign_mismatch_count"],
                "logit_max_abs_error": debug_validation["logit_max_abs_error"],
                "logit_mean_abs_error": debug_validation["logit_mean_abs_error"],
                "logit_sign_mismatch_count": debug_validation["logit_sign_mismatch_count"],
                "validation_tolerances": debug_validation["validation_tolerances"],
                "graph_summary": debug_validation["graph_summary"],
                "graph_target_tokens": debug_validation["graph_target_tokens"],
                "input_summary": input_summary,
                "drift_summary": {
                    "divergent_feature_count": drift_report["divergent_feature_count"],
                    "total_feature_count": drift_report["total_feature_count"],
                    "divergent_logit_count": drift_report["logit_summary"]["divergent_logit_count"],
                    "total_logit_count": drift_report["logit_summary"]["total_logit_count"],
                    "layer_with_max_divergence": drift_report["layer_with_max_divergence"],
                },
                "artifact_dir": debug_validation["artifact_dir"],
            },
            indent=2,
        )
    )

    display_layer_divergence_summary(drift_report["layer_summaries"])
    display_logit_drift_summary(drift_report["logit_summary"])

    key_logit_rows = [
        {
            "token_id": int(token_id),
            "token": label,
            "pre_logit": float(pre_logit),
            "post_logit": float(post_logit),
            "delta": float(post_logit - pre_logit),
        }
        for token_id, label, pre_logit, post_logit in zip(
            debug_validation["key_ids"],
            debug_validation["key_labels"],
            debug_validation["key_logits_pre"],
            debug_validation["key_logits_post"],
            strict=False,
        )
    ]
    display_diagnostic_rows("Key-token logit deltas", key_logit_rows)

    print("Intervention path comparison")
    print(json.dumps(debug_validation["intervention_paths"], indent=2))
    display_diagnostic_rows("Selected feature self-effect", [debug_validation["selected_feature_self_effect"]])
    display_diagnostic_rows("Same feature rows in graph", debug_validation["same_feature_rows_in_graph"])
    display_diagnostic_rows(
        f"Top {debug_validation['debug_validation_top_k']} activation error rows",
        debug_validation["activation_error_rows"],
    )
    display_diagnostic_rows("Activation layer summary", debug_validation["activation_layer_summary"])
    display_diagnostic_rows(
        f"Top {debug_validation['debug_validation_top_k']} expected-effect rows",
        debug_validation["expected_effect_rows"],
    )
    display_diagnostic_rows(
        f"Top {debug_validation['debug_validation_top_k']} actual-effect rows",
        debug_validation["actual_effect_rows"],
    )
    display_diagnostic_rows(
        f"Top {debug_validation['debug_validation_top_k']} logit error rows",
        debug_validation["logit_error_rows"],
    )

    if not debug_validation["all_passed"]:
        failure_message = (
            "Debug intervention validation exceeded configured tolerances; "
            "see RESULTS['debug_validation']."
        )
        if debug_validation["debug_validation_raise_on_failure"]:
            raise AssertionError(failure_message)
        print(f"WARNING: {failure_message}")
else:
    sweep_results = run_scale_sweep(
        embed_direction_result["direction"],
        embed_direction_result["group_a_ids"],
        embed_direction_result["group_b_ids"],
    )
    RESULTS["sweep_results"] = sweep_results
    for sweep_result in sweep_results:
        print(f"Scale={sweep_result['scale_factor']:.1f}x Δ={sweep_result['gap_delta']:+.4f}")
    scales = [entry["scale_factor"] for entry in sweep_results]
    deltas = [entry["gap_delta"] for entry in sweep_results]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(scales, deltas, "o-", color="#2471A3", linewidth=2, markersize=8)
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Scale Factor")
    ax.set_ylabel(f"Gap Δ ({embed_pipeline['target_a_tok']}−{embed_pipeline['target_b_tok']})")
    ax.set_title(f"Scale Factor Sweep — {MODEL_FAMILY}:{MODEL_VARIANT} {NOTEBOOK_CFG.analysis_concept_label}")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    for sweep_result in [sweep_results[0], sweep_results[-1]]:
        display_key_token_logits(
            sweep_result["pre_logits"],
            sweep_result["post_logits"],
            sweep_result["key_ids"],
            sweep_result["key_labels"],
            title=f"Embed {sweep_result['scale_factor']}× — Key-Token Logits",
        )

## Phase 7 and 8: Progressive Ablation and Sign-Aware Feature Selection

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("progressive ablation and sign-aware analysis are disabled in debug_intervention_pipelines mode.")
else:
    feature_pool = collect_feature_pool(
        embed_direction_result["direction"],
        embed_direction_result["group_a_ids"],
        embed_direction_result["group_b_ids"],
        top_n=max(max(ABLATION_N_LIST), TOP_N * 2),
    )

    RESULTS["feature_pool_summary"] = {
        "feature_count": int(feature_pool["feature_ids"].shape[0]),
        "target_a_id": feature_pool["target_a_id"],
        "target_b_id": feature_pool["target_b_id"],
    }

    ablation_groups, ablation_logit_diffs, ablation_results = run_ablations(feature_pool, embed_pipeline["pre_logits"])
    RESULTS["ablation_logit_diffs"] = ablation_logit_diffs

    display_ablation_chart(
        ablation_groups,
        logit_diffs=ablation_logit_diffs,
        title=f"Embed ablation — {MODEL_FAMILY}:{MODEL_VARIANT} {NOTEBOOK_CFG.analysis_concept_label}",
    )

    if ablation_results:
        strongest_n = max(ablation_results)
        display_key_token_logits(
            embed_pipeline["pre_logits"],
            ablation_results[strongest_n],
            feature_pool["key_ids"],
            feature_pool["key_labels"],
            title=f"Top-{strongest_n} Ablation — Key-Token Logits",
        )

    if ENABLE_SIGN_AWARE:
        sign_aware = run_sign_aware(feature_pool, embed_pipeline["pre_logits"])
        RESULTS["sign_aware"] = {
            "positive_feature_count": int(len(sign_aware["positive_features"])),
            "negative_feature_count": int(len(sign_aware["negative_features"])),
            "messages": list(sign_aware.get("messages", [])),
        }
        for message in sign_aware.get("messages", []):
            print(f"\n⚠️  Sign-aware: {message}")
        has_pos = len(sign_aware["positive_features"]) > 0
        has_neg = len(sign_aware["negative_features"]) > 0
        if has_pos or has_neg:
            feature_display = {}
            score_display = {}
            if has_pos:
                feature_display["Positive activation"] = feature_ids_to_tuples(sign_aware["positive_features"][:TOP_N])
                score_display["Positive activation"] = tensor_to_cpu(sign_aware["positive_scores"][:TOP_N]).tolist()
            if has_neg:
                feature_display["Negative activation"] = feature_ids_to_tuples(sign_aware["negative_features"][:TOP_N])
                score_display["Negative activation"] = tensor_to_cpu(sign_aware["negative_scores"][:TOP_N]).tolist()
            display_top_features_comparison(
                feature_display,
                score_display,
                neuronpedia_model=NEURONPEDIA_MODEL,
                neuronpedia_set=NEURONPEDIA_SET,
                neuronpedia_base_url=NOTEBOOK_CFG.neuronpedia_base_url,
            )
        else:
            print("\n⚠️  Sign-aware: No features had non-zero activations — nothing to display.")
        if "positive_post_logits" in sign_aware:
            display_key_token_logits(
                embed_pipeline["pre_logits"],
                sign_aware["positive_post_logits"],
                feature_pool["key_ids"],
                feature_pool["key_labels"],
                title=f"Positive features {DEFAULT_SCALE_FACTOR}× — Key-Token Logits",
            )
        if "negative_post_logits" in sign_aware:
            display_key_token_logits(
                embed_pipeline["pre_logits"],
                sign_aware["negative_post_logits"],
                feature_pool["key_ids"],
                feature_pool["key_labels"],
                title="Negative features ablated — Key-Token Logits",
            )
    else:
        print("\nSign-aware feature analysis: DISABLED (set ENABLE_SIGN_AWARE=true to enable)")

## Phase 9: Direction Consistency Probes

In [ ]:
if not USES_STORE_DIRECTION:
    print_phase_skip("direction consistency probes only apply when both embed and store directions are available.")
else:
    probe_results = run_direction_probes(embed_direction_result["direction"], store_direction_result["direction"])
    RESULTS["probe_results"] = probe_results
    for label, payload in probe_results.items():
        print(f"\n{label} direction probes:")
        print(f"{'Token':<15} {'Projection':>12} {'Group':>8}")
        print(f"{'-' * 15} {'-' * 12} {'-' * 8}")
        for row in payload["rows"]:
            print(f"{row['token']:<15} {row['projection']:>12.4f} {row['group']:>8}")
        separation = payload["mean_a"] - payload["mean_b"]
        print(f"Mean A: {payload['mean_a']:.4f}, Mean B: {payload['mean_b']:.4f}, Separation: {separation:.4f}")

## Phase 10: Local Explanation Coverage

In [ ]:
if IS_DEBUG_INTERVENTION_MODE:
    print_phase_skip("local explanation coverage is disabled in debug_intervention_pipelines mode.")
elif NOTEBOOK_CFG.check_local_explanation_coverage or NOTEBOOK_CFG.generate_missing_local_explanations:
    embed_feature_ids = embed_pipeline["feature_ids"][: NOTEBOOK_CFG.local_explanation_feature_limit]
    store_feature_ids = []
    if "store_pipeline" in RESULTS:
        store_feature_ids = store_pipeline["feature_ids"][: NOTEBOOK_CFG.local_explanation_feature_limit]
    explanation_preparation = prepare_local_explanation_backfill(
        NOTEBOOK_CFG,
        embed_feature_ids,
        store_feature_ids,
    )
    RESULTS["local_explanation_prefetch"] = {
        "export_roots": list(explanation_preparation.export_roots),
        "cache_dir": explanation_preparation.cache_dir,
        "checked_feature_urls": [feature_ref.feature_url for feature_ref in explanation_preparation.feature_refs],
        "missing_feature_urls": [
            status.feature_ref.feature_url
            for status in explanation_preparation.initial_statuses
            if not status.has_local_explanation
        ],
        "cache_ready_feature_urls": [
            status.feature_ref.feature_url
            for status in explanation_preparation.prefetch_statuses
            if status.cache_ready
        ],
        "prefetch_statuses": [
            {
                "feature_url": status.feature_ref.feature_url,
                "explanation_count": status.explanation_count,
                "cache_ready": status.cache_ready,
                "cache_source": status.cache_source,
                "cache_path": status.cache_path,
                "activation_rows": status.activation_rows,
                "error": status.error,
            }
            for status in explanation_preparation.prefetch_statuses
        ],
        "prefetch_failures": [
            {"feature_url": status.feature_ref.feature_url, "error": status.error}
            for status in explanation_preparation.prefetch_statuses
            if not status.cache_ready
        ],
    }
    print(json.dumps(RESULTS["local_explanation_prefetch"], indent=2))

    explanation_coverage = ensure_local_feature_explanations(
        explanation_preparation.feature_refs,
        generate_missing=NOTEBOOK_CFG.generate_missing_local_explanations,
        local_db_url=NOTEBOOK_CFG.local_neuronpedia_db_url,
        timeout_seconds=NOTEBOOK_CFG.local_explanation_timeout_seconds,
        type_name=NOTEBOOK_CFG.local_explanation_type_name,
        max_retries=NOTEBOOK_CFG.local_explanation_max_retries,
        retry_backoff_seconds=NOTEBOOK_CFG.local_explanation_retry_backoff_seconds,
    )
    RESULTS["local_explanation_coverage"] = {
        "use_localhost": NOTEBOOK_CFG.use_localhost,
        "neuronpedia_base_url": NOTEBOOK_CFG.neuronpedia_base_url,
        "explanation_type_name": NOTEBOOK_CFG.local_explanation_type_name,
        "timeout_seconds": NOTEBOOK_CFG.local_explanation_timeout_seconds,
        "max_retries": NOTEBOOK_CFG.local_explanation_max_retries,
        "retry_backoff_seconds": NOTEBOOK_CFG.local_explanation_retry_backoff_seconds,
        "checked_feature_urls": [status.feature_ref.feature_url for status in explanation_coverage.statuses],
        "missing_feature_urls": [
            status.feature_ref.feature_url
            for status in explanation_coverage.statuses
            if not status.has_local_explanation
        ],
        "generated_artifact_paths": [
            str(artifact.artifact_path) for artifact in explanation_coverage.generated_artifacts
        ],
        "generation_failures": [
            {"feature_url": failure.feature_ref.feature_url, "error": failure.error}
            for failure in explanation_coverage.generation_failures
        ],
        "prefetch_summary": RESULTS["local_explanation_prefetch"],
    }
    print(
        json.dumps(
            RESULTS["local_explanation_coverage"],
            indent=2,
        )
    )
else:
    print(
        "Local explanation coverage: DISABLED (set CHECK_LOCAL_EXPLANATION_COVERAGE=true or "
        "GENERATE_MISSING_LOCAL_EXPLANATIONS=true to enable)"
    )

## Cleanup

In [ ]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if EXPERIMENT_WORK_DIR is None:
    shutil.rmtree(WORK_ROOT, ignore_errors=True)

SUMMARY_RECORD = {
    "experiment_name": EXPERIMENT_NAME,
    "config_name": EXPERIMENT_CONFIG_NAME,
    "model_family": MODEL_FAMILY,
    "model_variant": MODEL_VARIANT,
    "model_name": MODEL_NAME,
    "concept_pair": concept_pair.name,
    "analysis_mode": NOTEBOOK_CFG.analysis_mode,
    "analysis_concept_label": NOTEBOOK_CFG.analysis_concept_label,
    "prompt_render_mode": NOTEBOOK_CFG.prompt_render_mode,
    "target_tokens": list(NOTEBOOK_CFG.target_tokens) if NOTEBOOK_CFG.target_tokens is not None else None,
    "target_token_ids": list(NOTEBOOK_CFG.target_token_ids) if NOTEBOOK_CFG.target_token_ids is not None else None,
    "work_root_removed": EXPERIMENT_WORK_DIR is None,
}

if "embed_pipeline" in RESULTS:
    SUMMARY_RECORD["embed_gap_delta"] = RESULTS["embed_pipeline"]["gap_delta"]
if "store_pipeline" in RESULTS:
    SUMMARY_RECORD["store_gap_delta"] = RESULTS["store_pipeline"]["gap_delta"]
if "comparison" in RESULTS:
    SUMMARY_RECORD["cosine_similarity"] = RESULTS["comparison"]["cosine_similarity"]
    SUMMARY_RECORD["feature_jaccard"] = RESULTS["comparison"]["feature_jaccard"]
if "store_direction_result" in RESULTS:
    SUMMARY_RECORD["prediction_correct"] = RESULTS["store_direction_result"]["prediction_info"]["n_correct"]
    SUMMARY_RECORD["prediction_total"] = RESULTS["store_direction_result"]["n_total"]
if "debug_validation" in RESULTS:
    SUMMARY_RECORD["debug_validation_passed"] = RESULTS["debug_validation"]["all_passed"]
    SUMMARY_RECORD["debug_activation_max_abs_error"] = RESULTS["debug_validation"]["activation_max_abs_error"]
    SUMMARY_RECORD["debug_logit_max_abs_error"] = RESULTS["debug_validation"]["logit_max_abs_error"]
    SUMMARY_RECORD["debug_selected_feature"] = list(RESULTS["debug_validation"]["selected_feature"])

print("Notebook complete.")
print(json.dumps(SUMMARY_RECORD, indent=2))